# NB-01 · Python Foundations for Health Data
## Module 01 — Health Analytics with Python
---
**Learning objectives**
- Set up a reproducible Python environment for health analytics
- Understand core Python data types used in health data workflows
- Read and inspect CSV, Excel, and JSON health data files
- Identify key health data sources: CMS, CDC, EHR exports
- Understand HIPAA-aware data handling basics

**Estimated time:** 1.5 hours  
**Level:** Beginner  
**Libraries:** `pandas`, `numpy`, `openpyxl`, `json`, `pathlib`


## 1. Environment Setup & Library Imports

In [ ]:
# Install any missing packages (run once)
# !pip install pandas numpy openpyxl

import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Library versions:")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print("Environment ready.")


## 2. Core Python Data Types in a Health Context

Health data pipelines use four foundational Python structures. Understanding them prevents the most common data-wrangling errors.


In [ ]:
# --- 2.1  Lists: ordered sequences (e.g. a patient's diagnosis codes) ---
diagnoses = ['E11.9', 'I10', 'N18.3', 'Z87.891']
print("Diagnoses list:", diagnoses)
print("First code :", diagnoses[0])
print("Last  code :", diagnoses[-1])
print("Slice [1:3]:", diagnoses[1:3])


In [ ]:
# --- 2.2  Dictionaries: key-value pairs (e.g. a single patient record) ---
patient = {
    'patient_id'  : 'PT-00142',
    'age'         : 67,
    'sex'         : 'F',
    'diagnoses'   : diagnoses,
    'los_days'    : 4,
    'payer'       : 'Medicare'
}

print("Patient record:")
for key, val in patient.items():
    print(f"  {key:15s}: {val}")


In [ ]:
# --- 2.3  Tuples: immutable pairs (e.g. ICD code + description) ---
icd_lookup = {
    'E11.9' : ('Type 2 diabetes mellitus', 'without complications'),
    'I10'   : ('Essential hypertension', ''),
    'N18.3' : ('Chronic kidney disease, stage 3', ''),
}

for code, (desc, note) in icd_lookup.items():
    suffix = f'  [{note}]' if note else ''
    print(f"  {code} → {desc}{suffix}")


In [ ]:
# --- 2.4  Sets: unique memberships (e.g. unique conditions in a cohort) ---
cohort_conditions_a = {'diabetes', 'hypertension', 'CKD', 'obesity'}
cohort_conditions_b = {'hypertension', 'heart failure', 'obesity', 'COPD'}

print("Shared conditions      :", cohort_conditions_a & cohort_conditions_b)
print("Only in cohort A       :", cohort_conditions_a - cohort_conditions_b)
print("All conditions (union) :", cohort_conditions_a | cohort_conditions_b)


## 3. Synthetic Health Dataset

We generate a realistic synthetic hospital discharge dataset so every learner starts from the same data without any data-use-agreement requirements.  
In real projects this would be replaced by a CMS, HCUP NIS, or EHR export.


In [ ]:
np.random.seed(42)
N = 500

# --- Controlled random draws to mimic real discharge distributions ---
ages      = np.random.normal(loc=62, scale=16, size=N).clip(18, 95).astype(int)
sexes     = np.random.choice(['M', 'F'], size=N, p=[0.48, 0.52])
payers    = np.random.choice(
    ['Medicare', 'Medicaid', 'Private', 'Self-pay', 'Other'],
    size=N, p=[0.40, 0.22, 0.28, 0.07, 0.03]
)
drg_codes = np.random.choice(
    [470, 291, 392, 690, 871, 193, 247, 603],
    size=N
)
drg_labels = {
    470:'Major joint replacement', 291:'Heart failure & shock',
    392:'Esophagitis/misc GI',    690:'Kidney/UTI',
    871:'Septicemia',             193:'Simple pneumonia',
    247:'Perc cardiovasc w stent',603:'Cellulitis'
}
los_days  = np.random.gamma(shape=2.5, scale=1.8, size=N).clip(1, 30).astype(int)
charges   = (los_days * np.random.uniform(1800, 4200, N)).round(2)
missing_mask = np.random.rand(N) < 0.08   # 8% missing charges (realistic)
charges[missing_mask] = np.nan

primary_dx = np.random.choice(
    ['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'],
    size=N
)
admit_type = np.random.choice(
    ['Emergency','Elective','Urgent','Newborn'],
    size=N, p=[0.52, 0.30, 0.16, 0.02]
)

df = pd.DataFrame({
    'patient_id'   : [f'PT-{str(i).zfill(5)}' for i in range(1, N+1)],
    'age'          : ages,
    'sex'          : sexes,
    'payer'        : payers,
    'drg_code'     : drg_codes,
    'drg_label'    : [drg_labels[d] for d in drg_codes],
    'primary_dx'   : primary_dx,
    'admit_type'   : admit_type,
    'los_days'     : los_days,
    'total_charge' : charges,
})

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(8)


## 4. Reading Health Data Files

In [ ]:
# --- 4.1  Save and re-read as CSV (most common EHR export format) ---
Path('/tmp/health_data').mkdir(exist_ok=True)
df.to_csv('/tmp/health_data/discharges.csv', index=False)

df_csv = pd.read_csv(
    '/tmp/health_data/discharges.csv',
    dtype={'patient_id': str, 'drg_code': int},
    parse_dates=False
)
print("CSV load — shape:", df_csv.shape)
print(df_csv.dtypes)


In [ ]:
# --- 4.2  Excel (common for hospital billing exports) ---
df.to_excel('/tmp/health_data/discharges.xlsx', index=False, sheet_name='Discharges')

df_xl = pd.read_excel(
    '/tmp/health_data/discharges.xlsx',
    sheet_name='Discharges',
    dtype={'patient_id': str}
)
print("Excel load — shape:", df_xl.shape)
df_xl.head(3)


In [ ]:
# --- 4.3  JSON (common for FHIR / API responses) ---
sample_fhir_like = [
    {
        'resourceType': 'Patient',
        'id': row['patient_id'],
        'gender': row['sex'].lower(),
        'age': int(row['age']),
        'condition': row['primary_dx']
    }
    for _, row in df.head(5).iterrows()
]

with open('/tmp/health_data/patients.json', 'w') as f:
    json.dump(sample_fhir_like, f, indent=2)

with open('/tmp/health_data/patients.json') as f:
    loaded = json.load(f)

df_json = pd.json_normalize(loaded)
print("JSON load — shape:", df_json.shape)
df_json


## 5. First-Pass Data Inspection

Always run these five commands before any analysis. They reveal dtype mismatches, hidden nulls, and cardinality surprises that will corrupt downstream results if ignored.


In [ ]:
# --- 5.1  Shape, dtypes, memory ---
print("Shape:", df.shape)
print()
df.info(memory_usage='deep')


In [ ]:
# --- 5.2  Head and tail ---
print("=== First 5 rows ===")
display(df.head())
print("\n=== Last 5 rows ===")
display(df.tail())


In [ ]:
# --- 5.3  Missing value audit ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0]
print("Columns with missing data:")
display(missing_df)


In [ ]:
# --- 5.4  Cardinality check (critical for categorical codes) ---
cat_cols = ['sex','payer','admit_type','primary_dx','drg_code']
for col in cat_cols:
    n_unique = df[col].nunique()
    top_vals = df[col].value_counts().head(3).to_dict()
    print(f"  {col:15s}: {n_unique:3d} unique | top-3: {top_vals}")


In [ ]:
# --- 5.5  Numeric summary with health-relevant percentiles ---
df[['age','los_days','total_charge']].describe(percentiles=[.10,.25,.50,.75,.90,.99])


## 6. Health Data Sources Overview

| Source | What it contains | Access | Typical format |
|--------|-----------------|--------|----------------|
| **CMS Medicare claims** | Inpatient, outpatient, Part D claims for Medicare beneficiaries | Limited dataset (DUA required) | CSV / SAS |
| **HCUP NIS** | National inpatient sample, ~7M stays/year, all payers | AHRQ purchase or academic agreement | SAS / CSV |
| **CDC BRFSS** | Behavioural risk factors, chronic conditions, state-level | Public download | XPT / CSV |
| **PhysioNet MIMIC-IV** | De-identified ICU data: labs, vitals, meds, notes | Free after credentialing (~30 min) | CSV / Parquet |
| **Synthea** | Fully synthetic FHIR patients, no DUA needed | Open source, run locally | FHIR JSON / CSV |
| **NYC SPARCS** | NY State inpatient discharges with ZIP codes | Public download | CSV |

> **HIPAA reminder:** Real patient data requires a data use agreement, IRB approval, and safe-harbour or expert-determination de-identification before analysis. Use Synthea or public demo datasets for learning.


## 7. ICD-10 & DRG Code Structures

In [ ]:
# ICD-10-CM structure: Category . Etiology/Anatomic site . Severity
def parse_icd10(code):
    code = code.replace('.','')
    category  = code[:3]
    if len(code) > 3:
        subcategory = code[3]
        detail = code[4:] if len(code) > 4 else ''
    else:
        subcategory, detail = '', ''
    return {'category': category, 'subcategory': subcategory, 'detail': detail}

for icd in ['E11.9','I10','N18.3','A41.9']:
    parsed = parse_icd10(icd)
    print(f"  {icd:8s} → {parsed}")


In [ ]:
# DRG distribution in our dataset
drg_dist = (
    df.groupby(['drg_code','drg_label'])
      .agg(n_cases=('patient_id','count'),
           mean_los=('los_days','mean'),
           mean_charge=('total_charge','mean'))
      .sort_values('n_cases', ascending=False)
      .round(1)
)
print("DRG distribution:")
display(drg_dist)


## 8. Knowledge Check

Answer these questions using the dataset before moving to NB-02.

1. What percentage of patients are `Medicare` beneficiaries?
2. Which DRG has the highest mean length of stay?
3. How many patients have a missing `total_charge`?
4. What is the 90th-percentile LOS in days?
5. List all unique `admit_type` values and their frequencies.


In [ ]:
# --- Exercise space ---
# 1. Medicare share
medicare_pct = (df['payer'] == 'Medicare').mean() * 100
print(f"1. Medicare share: {medicare_pct:.1f}%")

# 2. Highest mean LOS by DRG
print("\n2. Highest mean LOS by DRG:")
display(df.groupby('drg_label')['los_days'].mean().sort_values(ascending=False).head(3))

# 3. Missing charges
print(f"\n3. Missing total_charge: {df['total_charge'].isnull().sum()}")

# 4. 90th percentile LOS
print(f"\n4. 90th-percentile LOS: {df['los_days'].quantile(0.90):.0f} days")

# 5. Admit type frequencies
print("\n5. Admit type frequencies:")
display(df['admit_type'].value_counts())


---
## Summary

In this notebook you:
- Built the standard Python health analytics environment
- Reviewed four core data types with clinical examples
- Generated and inspected a synthetic hospital discharge dataset
- Loaded health data from CSV, Excel, and JSON formats
- Performed a first-pass data audit (shape, dtypes, missingness, cardinality)
- Mapped the key public health data sources and their access requirements
- Decoded ICD-10 and DRG code structures

**Next:** `NB-02 · Descriptive Statistics & Health Metrics` — prevalence, incidence, rates, and stratified summaries.
